In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [ ]:
# Load CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

# Preprocess and normalize data
train_images = train_images / 255.0
test_images = test_images / 255.0
train_labels = to_categorical(train_labels, num_classes=10)
test_labels = to_categorical(test_labels, num_classes=10)


# The to_categorical function from Keras' utils module is used to convert integer class labels into one-hot encoded vectors.
# In the context of classification tasks, the model outputs a probability distribution over the classes,
# so the labels are encoded as binary vectors with a 1 at the corresponding class index and 0s elsewhere.
# For example, if you have 10 classes, the class label 2 would be converted to [0, 0, 1, 0, 0, 0, 0, 0, 0, 0].

In [ ]:
train_images.shape

(50000, 32, 32, 3)

In [ ]:
# Load pre-trained ResNet-101 model (excluding top classification layer)
base_model = ResNet101(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

#include_top=False  >> remove all fully connected layers not only softmax

171446536/171446536 [==============================] - 1s 0us/step


In [ ]:
# Add custom top layers for CIFAR-10 classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(10, activation='softmax')(x)

In [ ]:
# Create the model
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train the model
batch_size = 64
epochs = 5
model.fit(train_images, train_labels, batch_size=batch_size, epochs=epochs, validation_data=(test_images, test_labels))

Epoch 1/5
782/782 [==============================] - 162s 93ms/step - loss: 1.3328 - accuracy: 0.5482 - val_loss: 2.0911 - val_accuracy: 0.3223
Epoch 2/5
782/782 [==============================] - 71s 91ms/step - loss: 1.0986 - accuracy: 0.6217 - val_loss: 19265.0410 - val_accuracy: 0.1000
Epoch 3/5
782/782 [==============================] - 71s 91ms/step - loss: 1.1076 - accuracy: 0.6128 - val_loss: 1.0747 - val_accuracy: 0.6221
Epoch 4/5
782/782 [==============================] - 70s 90ms/step - loss: 0.9264 - accuracy: 0.6806 - val_loss: 1.7486 - val_accuracy: 0.4673
Epoch 5/5
782/782 [==============================] - 71s 90ms/step - loss: 0.8748 - accuracy: 0.6965 - val_loss: 0.9273 - val_accuracy: 0.6780


In [ ]:
# Evaluate the model
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test accuracy: {test_acc}")

313/313 [==============================] - 7s 19ms/step - loss: 0.9273 - accuracy: 0.6780
Test accuracy: 0.6779999732971191
